# 22 - Do my ideas have ground to stand on? Hospitals made to differ ON PURPOSE

**Pre-registration:** `docs/2026-11-02-shift-test-preregistration.md` (read first). Background: notebook 21 found that on the natural splits
even an oracle that averages exactly within the true group does not beat FedAvg + fine-tune (the hospitals are too similar).

Two ways to force a difference, each with a strength dial (pick any in cell 3):
- **Idea 1, concept shift** (`shift10/25/50/100`): the later-era hospitals get the labels of that % of antibiotics INVERTED, so the two groups of 5
  hospitals follow conflicting rules. FedAvg averages conflicting rules; a method that groups similar hospitals should not.
- **Idea 2, different test panels** (`panel50`, `panel20`): each group of 5 hospitals is only tested on its own antibiotic panel (50% / 20% shared).
  Labels untouched; the graphs differ. This is where the topology fingerprint should be able to see something.

**Arms** (same 8 rounds + 8-epoch fine-tune for all, so no budget confound): FedAvg + fine-tune (the bar), v3 + antibiogram fingerprint (the method),
oracle (true group = the best any grouping could do). `v3_topo` joins on the panel experiments; the **shuffled** control is run only if v3_ab wins.
No local-only, SimFT, FedProx etc.

**Runtime (T4, 3 seeds):** about 9 runs per concept experiment (a short test), 12 per panel experiment. Strongest shift first;
**if the oracle cannot beat FedAvg + fine-tune at `shift100`, stop**: the method cannot win anywhere easier.

In [ ]:
# 1) Environment: Colab (clone + install + mount Drive) or local (.venv)
import os, sys
ON_COLAB = 'google.colab' in sys.modules
BRANCH = 'feat/budget-and-era'
if ON_COLAB:
    !git clone -b main https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout main && git pull)
    !cd /content/Capstone-amr-fed && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
    !cd /content/Capstone-amr-fed && git log -1 --oneline
    !pip install -q torch_geometric 'flwr[simulation]' networkx scipy
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # edit to your folder if different
else:
    os.environ.setdefault('ARMD_DIR', '/Users/vikramadityasharma/Library/CloudStorage/GoogleDrive-vikramaditya.raunak@gmail.com/.shortcut-targets-by-id/1uVx5PmldPyJBMtnB9Lkl9N1rNBQrmJt-/ARMD')
import torch, ray, flwr, torch_geometric
print('colab:', ON_COLAB, '| python', sys.version.split()[0], '| torch', torch.__version__, '| flwr', flwr.__version__,
      '| ray', ray.__version__, '| device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('ARMD_DIR:', os.environ['ARMD_DIR'])

In [ ]:
# 2) Imports + sanity
from pathlib import Path
if ON_COLAB:
    SRC = Path('/content/Capstone-amr-fed/src')
else:
    SRC = next(p / 'src' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'amr_fed').is_dir())
sys.path.insert(0, str(SRC))
from amr_fed import config
D = Path(config.DATA_DIR)
assert D.exists(), 'ARMD_DIR is wrong - fix cell 1 and re-run.'
from amr_fed.federated import budget_runs as BR, shift_runs as S, final_protocol as P
from amr_fed.data_loader import load_cohort_frame, PK
print('experiments:', S.EXPERIMENTS)

In [ ]:
# 3) What this session runs. Edit EXPERIMENTS (strongest first) and TAG (use a different TAG per Colab account).
import json, ray
os.environ.pop('AMR_RNG_PROBE', None)
TAG = 'S'
EXPERIMENTS = ['shift100', 'shift50', 'shift25', 'shift10', 'panel20', 'panel50']   # any subset, any order
SEEDS = (42, 43, 44)
EXTRA_ARMS = []            # add 'v3_ab_shuffled' once v3_ab beats FedAvg + fine-tune (cell 6 tells you)
RES_DIR = SRC.parent / 'results'; RES_DIR.mkdir(exist_ok=True)

def paths(stem):
    local = RES_DIR / f'{stem}_{TAG}.json'
    mirror = Path(f'/content/drive/MyDrive/{stem}_{TAG}.json') if ON_COLAB else None
    return local, mirror

def load(stem):
    local, mirror = paths(stem)
    src = local if local.exists() else (mirror if mirror and mirror.exists() else None)
    return json.load(open(src)) if src else {}

df = load_cohort_frame()
print('rows:', f'{len(df):,}')

In [ ]:
# 4) Run (resume-safe, checkpointed after every seed of every arm; re-run this cell after a disconnect)
SPECS = S.arm_specs()
for name in EXPERIMENTS:
    stem = f'amr_shift_{name}'
    res = load(stem)
    dfx, part = S.build(name, df)
    arms = {k: SPECS[k] for k in (*S.DEFAULT_ARMS[S.kind(name)], *EXTRA_ARMS)}
    kw = dict(seeds=SEEDS, df=dfx, rounds=8, local_epochs=BR.LOCAL_EPOCHS, local_only_epochs=40, hidden=128,
              partition_fn=part, n_clients=10)
    print(f'\n######## {name}: {len(dfx):,} rows, arms {list(arms)}')
    res = BR.run_part(res, arms, prefix=f'shift/{name}', kw=kw,
                      checkpoint=lambda r, stem=stem: P.save_json(r, *paths(stem)), ray_shutdown=ray.shutdown)
    P.save_json(res, *paths(stem))
    del dfx

In [ ]:
# 5) GATE report per experiment: oracle - FedAvg+ft is the headroom; v3_ab - FedAvg+ft and v3_ab - shuffled are the claims
for name in EXPERIMENTS:
    res = load(f'amr_shift_{name}')
    print(f'\n################ {name} ################')
    print(S.report(res))
    groups = {s: S.client_groups(name, df, s) for s in SEEDS}
    print('weights (mean over hospitals and seeds):'); print(S.weights(res, groups))

**How to read cell 5 (fixed in advance, see the pre-registration):**
1. `oracle+ft - fedavg+ft` CI above zero -> there is headroom, similarity can matter at this shift. Not above zero -> stop, report "no headroom".
2. Only if (1) holds: `v3_ab+ft - fedavg+ft` above zero -> the fingerprint recovers part of the headroom. Then set `EXTRA_ARMS = ['v3_ab_shuffled']`, re-run cell 4
   and check `v3_ab+ft - v3_ab_shuffled+ft` above zero (otherwise the gain is just "weight yourself more").
3. The weights show whether v3_ab put its weight on the true group (own group high, other group low).

In [ ]:
# 6) Merge two accounts' files (put both in results/ or Drive), then set TAG = 'merged' and re-run cell 5
import glob
for name in S.EXPERIMENTS:
    stem = f'amr_shift_{name}'
    parts = sorted(glob.glob(str(RES_DIR / f'{stem}_[!m]*.json')))
    if ON_COLAB: parts += sorted(glob.glob(f'/content/drive/MyDrive/{stem}_[!m]*.json'))
    if parts:
        merged = P.merge_results(parts)
        P.save_json(merged, RES_DIR / f'{stem}_merged.json')
        print(stem, 'merged arms:', [a for a in merged if not a.startswith('_')], 'from', len(parts), 'files')